# August 2026 verification · **00 — Sources**

**Why this pipeline exists.** I want to write about the August 2026 RTT
release: record elective activity, yet a growing over-52-week tail, and
7.37m pathways against 6.24m patients. Every figure I had came from the BMA's
compilation and from Healthcare Management, because NHS England's pages were
serving stale content when I last looked. I will not write off a secondary
source, so this pipeline rebuilds every headline from NHS England's own files
first.

This notebook does no work. It just puts on record:

1. which files I read, and that every one is a **primary** NHS England output;
2. the claims under test, quoted as the secondary sources state them;
3. the measure I use, and why it is the same one as in my earlier RTT pieces.

In [1]:
# I keep the catalogue and the claims in src/ so every notebook reads the same
# list, and nothing gets retyped by hand.
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from sources import SOURCES   # noqa: E402
from claims import CLAIMS     # noqa: E402

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

## 1 — The files

All of them were copied off england.nhs.uk's statistical work area pages on
16 October 2026. `primary` means NHS England's own output, not a compilation.

In [2]:
rows = []
for s in SOURCES:
    for filename, url, desc in s["files"]:
        rows.append({"slug": s["slug"], "file": filename, "vintage": s["vintage"],
                     "primary": s["primary"], "reuse_from_two_maps": bool(s["reuse_from"]),
                     "what_it_is": desc})
catalogue = pd.DataFrame(rows)
# I refuse to carry on if anything in the catalogue is not a primary source.
assert catalogue.primary.all(), "a non-primary source has crept into the catalogue"
print(f"{len(catalogue)} files, all primary NHS England outputs")
catalogue

33 files, all primary NHS England outputs


,slug,file,vintage,primary,reuse_from_two_maps,what_it_is
0,nhse_rtt_2025-04,Full-CSV-data-file-Apr25-ZIP-4M-revised.zip,revised 12 Feb 2026,True,False,Full extract: provider x commissioner x RTT part x treatment function x weekly wait bands (0-104+).
1,nhse_rtt_2025-05,Full-CSV-data-file-May25-ZIP-4M-revised.zip,revised 12 Feb 2026,True,False,Full extract: provider x commissioner x RTT part x treatment function x weekly wait bands (0-104+).
2,nhse_rtt_2025-06,Full-CSV-data-file-Jun25-ZIP-4M-revised.zip,revised 12 Feb 2026,True,False,Full extract: provider x commissioner x RTT part x treatment function x weekly wait bands (0-104+).
3,nhse_rtt_2025-07,Full-CSV-data-file-Jul25-ZIP-4M-revised-2.zip,"revised 13 Nov 2025, again 12 Feb 2026",True,False,Full extract: provider x commissioner x RTT part x treatment function x weekly wait bands (0-104+).
4,nhse_rtt_2025-08,Full-CSV-data-file-Aug25-ZIP-4M-revised-2.zip,"revised 13 Nov 2025, again 12 Feb 2026",True,False,Full extract: provider x commissioner x RTT part x treatment function x weekly wait bands (0-104+).
5,nhse_rtt_2025-09,Full-CSV-data-file-Sep25-ZIP-4M-revised.zip,revised 12 Feb 2026,True,False,Full extract: provider x commissioner x RTT part x treatment function x weekly wait bands (0-104+).
6,nhse_rtt_2025-10,Full-CSV-data-file-Oct25-ZIP-4M-SrRW6y.zip,as first published,True,False,Full extract: provider x commissioner x RTT part x treatment function x weekly wait bands (0-104+).
7,nhse_rtt_2025-11,Full-CSV-data-file-Nov25-ZIP-4M-1Xmjkk.zip,as first published,True,False,Full extract: provider x commissioner x RTT part x treatment function x weekly wait bands (0-104+).
8,nhse_rtt_2025-12,Full-CSV-data-file-Dec25-ZIP-3M-revised.zip,revised 9 Jul 2026,True,False,Full extract: provider x commissioner x RTT part x treatment function x weekly wait bands (0-104+).
9,nhse_rtt_2026-01,Full-CSV-data-file-Jan26-ZIP-4M-WL5BiP.zip,as first published,True,False,Full extract: provider x commissioner x RTT part x treatment function x weekly wait bands (0-104+).


## 2 — The claims under test

Nothing downstream reads these numbers. They exist so the gate can put each
one beside what the primary files say.

In [3]:
claims = pd.DataFrame(CLAIMS, columns=["key", "claimed", "unit", "as_quoted", "source"])
claims

,key,claimed,unit,as_quoted,source
0,rtt_incomplete_pathways,7370000.0,pathways,7.37 million pathways,BMA / Healthcare Management
1,rtt_unique_patients,6240000.0,patients,6.24 million patients,BMA / Healthcare Management
2,rtt_change_on_july,41848.0,pathways,"up 41,848 on July",Healthcare Management
3,rtt_over_52_weeks,113000.0,pathways,"~113,000 waiting more than a year",BMA
4,rtt_over_52_change,7000.0,pathways,"+7,000 over 52 weeks (article title)",article brief
5,rtt_over_52_pct_change,6.0,%,over-52-week group grew by roughly six per cent,article brief
6,rtt_total_pct_change,1.0,% (upper bound),list grew by well under one per cent,article brief
7,rtt_median_weeks,12.2,weeks,median wait 12.2 weeks,BMA
8,cwt_62d_pct,71.9,%,62-day performance 71.9 per cent,BMA
9,cwt_fds_pct,78.6,%,28-day faster diagnosis 78.6 per cent,BMA


## 3 — The measure, and why it matters

My earlier RTT pieces (Two Maps and the others) use the **reported-provider**
measure: incomplete pathways summed across the providers that submitted,
using the `C_999` treatment-function totals, with commissioner code `NONC` rows
excluded. That is exactly what NHS England's *Incomplete Provider* workbook
publishes. In July 2026 it came to **7,131,685** pathways and **108,263** over
52 weeks.

NHS England's **headline** (the press notice) is a different number. It adds
estimates for trusts that did not submit. In July that put the over-52-week
figure at 111,379 rather than 108,263. The BMA's 7.37m looks like a
headline-basis figure, so I have to rebuild **both** measures and say which
is which:

| Gate | Measure | Target |
|---|---|---|
| A | reported-provider, NONC excluded (**my primary measure**) | Incomplete Provider Aug26 workbook, provider by provider, exact |
| A′ | the same, July 2026 | Two Maps' 7,131,685 / 108,263, exact. This proves it is the same measure |
| B | headline = reported + NHS England's missing-trust estimates | Overview time series and press notice |
| C | Cancer Waiting Times, provider basis, Within / Total | statistical release and national time series |

Patients (6.24m) are not in any provider file. RTT is a pathway count. So
that figure can only be **sourced** from the press notice, not rebuilt, and I
label it that way.